**Objetivo:** Validar as matrizes de conectividade (shape, NaN, simetria, intervalo) e a correspondência com os participantes.

**Entradas:** `Data/conectivity/connectivity_matrices_abide.pkl` (notebook 04), `Data/raw/phenotypic_abide.csv`.

**Processamento:** Verificações por matriz; exclusão de sujeitos com problemas.

**Saídas:** `Data/conectivity/connectivity_matrices_abide_validado.pkl`; `Docs/05_relatorio_exclusoes.csv` (somente se houver exclusões).

In [ ]:
# ------------------------------------------------------------
# Configuração do projeto (execução local — VS Code)
# Garante que a raiz do projeto esteja no sys.path, funcionando tanto se o
# diretório de trabalho do kernel for notebooks/ quanto a raiz do projeto.
# ------------------------------------------------------------
import sys
from pathlib import Path

_raiz = Path.cwd().resolve()
for _cand in (_raiz, *_raiz.parents):
    if (_cand / 'src' / 'config.py').exists():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a pasta src/config.py a partir de '
        f'{Path.cwd()}. Abra a pasta TCC_TEA_TDAH/ no VS Code.'
    )
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import os
import random
import numpy as np

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()

random.seed(SEED)
np.random.seed(SEED)

print('Projeto (BASE_DIR) =', BASE_DIR)

In [2]:
exigir_arquivo(PATHS['conectivity'] / 'connectivity_matrices_abide.pkl', '04_Conectividade_ABIDE.ipynb')
exigir_arquivo(PATHS['raw'] / 'phenotypic_abide.csv', '02_Download_ABIDE.ipynb')

import pickle
import numpy as np
import pandas as pd

with open(os.path.join(PATHS['conectivity'], 'connectivity_matrices_abide.pkl'), 'rb') as f:
    conn_dict = pickle.load(f)

phenotypic = pd.read_csv(os.path.join(PATHS['raw'], 'phenotypic_abide.csv'))
phenotypic = phenotypic.set_index('SUB_ID')

print(f'{len(conn_dict)} sujeitos carregados para validação.')

871 sujeitos carregados para validação.


#### Validação da Matriz

In [3]:
def validar_matriz(mat, esperado_shape=(200, 200), tol=1e-6):
    '''Retorna uma lista de problemas encontrados na matriz, vazia se OK.'''
    problemas = []
    if mat.shape != esperado_shape:
        problemas.append(f'shape inesperado: {mat.shape}')
    if np.isnan(mat).any():
        problemas.append('contém NaN')
    if not np.allclose(mat, mat.T, atol=tol):
        problemas.append('matriz não é simétrica')
    if mat.min() < -1 - tol or mat.max() > 1 + tol:
        problemas.append(f'valores fora de [-1,1]: min={mat.min():.3f}, max={mat.max():.3f}')
    return problemas

relatorio = []
sujeitos_validos = {}

for sub_id, mat in conn_dict.items():
    problemas = validar_matriz(mat)
    site = phenotypic.loc[sub_id, 'SITE_ID'] if sub_id in phenotypic.index else 'desconhecido'
    if problemas:
        relatorio.append({'SUB_ID': sub_id, 'SITE_ID': site, 'problemas': '; '.join(problemas)})
    else:
        sujeitos_validos[sub_id] = mat

print(f'Sujeitos válidos: {len(sujeitos_validos)} / {len(conn_dict)}')
print(f'Sujeitos excluídos: {len(relatorio)}')

Sujeitos válidos: 871 / 871
Sujeitos excluídos: 0


In [4]:
df_relatorio = pd.DataFrame(relatorio)
if not df_relatorio.empty:
    print(df_relatorio['SITE_ID'].value_counts())
    df_relatorio.to_csv(os.path.join(PATHS['docs'], '05_relatorio_exclusoes.csv'), index=False)
else:
    print('Nenhum sujeito excluído — todas as matrizes passaram na validação.')

Nenhum sujeito excluído — todas as matrizes passaram na validação.


In [5]:
# Salva apenas os dados validados — este é o dataset "limpo" que segue
# para as próximas etapas (análise exploratória e preparação final)
out_path = os.path.join(PATHS['conectivity'], 'connectivity_matrices_abide_validado.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(sujeitos_validos, f)

print('Dados validados salvos em:', out_path)

Dados validados salvos em: <BASE_DIR>/Data/conectivity/connectivity_matrices_abide_validado.pkl
